In [5]:
"""
Digikala comments cleaning — v2
Fixes applied vs. the original notebook:
  1. Check/dedupe products['id'] BEFORE merging (root cause of the
     ~2.93M "duplicate rows" found after merging).
  2. Drop duplicate comment ids as a separate, independent safety net.
  3. Treat rate == 0.0 as a missing-value sentinel (not a real rating) —
     evidenced by ~570k "recommended" reviews having rate == 0.00.
  4. Drop the single rate == 2500.0 data-entry error.
  5. Tighten the cross-product spam rule so it stops flagging common,
     genuine short reviews ("عالی", "خوب بود") as spam.
  6. Fold in the near-duplicate brand spellings you already discovered.
  7. Recompute skew/naive-baseline stats only after cleaning.
  8. Stratified sampling without the pandas groupby-apply FutureWarning.
"""

import re
import ast
import numpy as np
import pandas as pd
from scipy.stats import skew

# ------------------------------------------------------------------
# 0. Load + verify merge integrity BEFORE anything else
# ------------------------------------------------------------------
comments = pd.read_csv("/kaggle/input/datasets/areforumiehei/digikala-comments/digikala-comments.csv")
products = pd.read_csv("/kaggle/input/datasets/areforumiehei/digikala-products/digikala-products.csv")

dup_product_ids = products['id'].duplicated().sum()
print(f"Duplicate product ids in products table: {dup_product_ids:,}")
if dup_product_ids > 0:
    before_p = len(products)
    products = products.drop_duplicates(subset='id', keep='first')
    print(f"  -> dropped {before_p - len(products):,} duplicate product rows before merging")

merged = comments.merge(
    products[['id', 'Category1', 'Category2', 'sub_category', 'Brand']],
    left_on='product_id', right_on='id', how='left', suffixes=('', '_product')
).drop(columns=['id_product'])
print(f"Rows after merge: {len(merged):,}")

/tmp/ipykernel_58/2933161656.py:27: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  products = pd.read_csv("/kaggle/input/datasets/areforumiehei/digikala-products/digikala-products.csv")


Duplicate product ids in products table: 335,144
  -> dropped 335,144 duplicate product rows before merging
Rows after merge: 6,156,289


In [6]:
# ------------------------------------------------------------------
# 1. Deduplicate comments themselves (independent safety net)
# ------------------------------------------------------------------
before = len(merged)
merged = merged.drop_duplicates(subset='id', keep='first').reset_index(drop=True)
print(f"Dropped {before - len(merged):,} duplicate comment ids "
      f"(was 2,931,984 before fixing the merge — should be much smaller now)")

Dropped 3,229 duplicate comment ids (was 2,931,984 before fixing the merge — should be much smaller now)


In [7]:
# ------------------------------------------------------------------
# 2. Fix the 'rate' column
# ------------------------------------------------------------------
# a) drop the data-entry error
n_bad_range = (merged['rate'] > 5).sum()
merged = merged[merged['rate'] <= 5].copy()
print(f"Dropped {n_bad_range:,} rows with rate > 5 (e.g. the 2500.0 outlier)")

# b) rate == 0.0 looks like "no score given", not a real rating
zero_but_recommended = ((merged['rate'] == 0) &
                         (merged['recommendation_status'] == 'recommended')).sum()
print(f"rate==0 rows labeled 'recommended': {zero_but_recommended:,} "
      f"-> treating rate==0 as missing, not a real 0-star rating")

merged['rate_clean'] = merged['rate'].replace(0.0, np.nan)
missing_rate = merged['rate_clean'].isna().sum()
merged = merged.dropna(subset=['rate_clean']).copy()
print(f"Dropped {missing_rate:,} rows with missing/invalid rate")
print(f"Remaining rows: {len(merged):,}")

# NOTE: rate is a continuous ~0-5 score (178 distinct levels seen), not a
# discrete 1-5 star pick. Confirm this against Digikala's own docs/UI and
# state it explicitly in your paper's dataset description.

Dropped 1 rows with rate > 5 (e.g. the 2500.0 outlier)
rate==0 rows labeled 'recommended': 425,680 -> treating rate==0 as missing, not a real 0-star rating
Dropped 529,021 rows with missing/invalid rate
Remaining rows: 5,624,038


In [8]:
# ------------------------------------------------------------------
# 3. Text quality flags
# ------------------------------------------------------------------
def token_count(text):
    return 0 if pd.isna(text) else len(str(text).strip().split())

merged['body_token_count'] = merged['body'].apply(token_count)
merged['is_ultra_short'] = merged['body_token_count'] < 2

PERSIAN_RE = re.compile(r'[\u0600-\u06FF]')
HTML_RE = re.compile(r'<[^>]+>')
PLACEHOLDER_VALUES = {'-', '.', '،', '؟', '!', '_', '؛', ''}

def text_flag(text):
    if pd.isna(text):
        return 'null'
    t = str(text).strip()
    if t in PLACEHOLDER_VALUES:
        return 'placeholder'
    if HTML_RE.search(t):
        return 'html_fragment'
    if not PERSIAN_RE.search(t):
        return 'no_persian_chars'
    return 'ok'

merged['body_text_flag'] = merged['body'].apply(text_flag)

URL_RE = re.compile(r'(https?://|www\.)\S+')
PHONE_RE = re.compile(r'(\+?\d[\d\-\s]{7,}\d)')
PROMO_KEYWORDS = ['تخفیف ویژه', 'کد تخفیف', 'کانال تلگرام', 'اینستاگرام']

def is_promo_spam(text):
    if pd.isna(text):
        return False
    t = str(text)
    if URL_RE.search(t) or PHONE_RE.search(t):
        return True
    return any(kw in t for kw in PROMO_KEYWORDS)

merged['is_promo_spam'] = merged['body'].apply(is_promo_spam)

# --- FIXED cross-product spam check ---
# Original rule (reused across >3 products) flagged 2,876,768 rows —
# almost entirely short generic-but-genuine reviews. Require a length
# floor AND a much higher reuse threshold so only real copy-paste text
# gets caught.
SPAM_MIN_TOKENS = 6
SPAM_MIN_PRODUCTS = 8

candidate_bodies = merged[(merged['body_text_flag'] == 'ok') &
                           (merged['body_token_count'] >= SPAM_MIN_TOKENS)]
reuse_counts = candidate_bodies.groupby('body')['product_id'].nunique()
cross_product_spam = reuse_counts[reuse_counts > SPAM_MIN_PRODUCTS]

merged['is_cross_product_dupe'] = (
    (merged['body_token_count'] >= SPAM_MIN_TOKENS) &
    merged['body'].isin(cross_product_spam.index)
)
print(f"Cross-product spam (tightened rule): {merged['is_cross_product_dupe'].sum():,} "
      f"(was 2,876,768 with the original loose rule)")

quality_cols = ['is_ultra_short', 'is_promo_spam', 'is_cross_product_dupe']
merged['any_quality_flag'] = merged[quality_cols].any(axis=1)
print(merged[quality_cols + ['any_quality_flag']].sum())

Cross-product spam (tightened rule): 42,735 (was 2,876,768 with the original loose rule)
is_ultra_short           436127
is_promo_spam              3993
is_cross_product_dupe     42735
any_quality_flag         482764
dtype: int64


In [9]:
# ------------------------------------------------------------------
# 4. advantages / disadvantages parsing (unchanged — this was correct)
# ------------------------------------------------------------------
def parse_list_string(text):
    if pd.isna(text):
        return None
    t = str(text).strip()
    if t in ('', '[]', "['']", 'nan'):
        return []
    try:
        parsed = ast.literal_eval(t)
        if isinstance(parsed, list):
            return [item.strip() for item in parsed if item and item.strip()]
    except (ValueError, SyntaxError):
        pass
    return [t]

merged['advantages_list'] = merged['advantages'].apply(parse_list_string)
merged['disadvantages_list'] = merged['disadvantages'].apply(parse_list_string)
merged['n_advantages'] = merged['advantages_list'].apply(lambda x: len(x) if x else 0)
merged['n_disadvantages'] = merged['disadvantages_list'].apply(lambda x: len(x) if x else 0)

In [10]:
# ------------------------------------------------------------------
# 5. Category/Brand normalization + fold in known near-duplicates
# ------------------------------------------------------------------
def normalize_text(text):
    if pd.isna(text):
        return None
    t = str(text).strip()
    if t in ('', '-', 'نامشخص', 'سایر', 'متفرقه'):
        return None
    t = t.replace('ي', 'ی').replace('ك', 'ک')
    t = t.replace('\u200c', ' ').replace('\xa0', ' ')
    t = re.sub(r'\s+', ' ', t).strip()
    return t

for col in ['Category1', 'Category2', 'sub_category', 'Brand']:
    merged[f'{col}_clean'] = merged[col].apply(normalize_text)

# Manually fold the spacing-variant duplicates you already found
BRAND_MERGE_MAP = {
    'بی ول': 'بیول', 'بای لندو': 'بایلندو', 'میتو': 'می تو',
    'هایدنت': 'های دنت', 'نوآکنه': 'نو آکنه',
    'انتشارات مهر اندیش': 'انتشارات مهراندیش', 'بی بی جم': 'بیبی جم',
    'بادی گارد': 'بادیگارد', 'آی مکس': 'آیمکس', 'بی تا': 'بیتا',
}
merged['Brand_clean'] = merged['Brand_clean'].replace(BRAND_MERGE_MAP)

In [11]:
# ------------------------------------------------------------------
# 6. Recompute distribution stats on the CLEANED target only
# ------------------------------------------------------------------
print(f"\nFinal row count: {len(merged):,}")
print(merged['rate_clean'].describe())
print(f"Skewness (should now be small, not 323.71): {skew(merged['rate_clean']):.3f}")

mode_rate = merged['rate_clean'].mode()[0]
mean_rate = merged['rate_clean'].mean()
naive_mode_mae = (merged['rate_clean'] - mode_rate).abs().mean()
naive_mean_mae = (merged['rate_clean'] - mean_rate).abs().mean()
naive_mean_rmse = np.sqrt(((merged['rate_clean'] - mean_rate) ** 2).mean())
print(f"Naive mode-baseline MAE: {naive_mode_mae:.3f}")
print(f"Naive mean-baseline MAE/RMSE: {naive_mean_mae:.3f} / {naive_mean_rmse:.3f}")
print("Any model you build should meaningfully beat these.")



Final row count: 5,624,038
count    5.624038e+06
mean     3.990933e+00
std      1.147157e+00
min      5.000000e-02
25%      3.000000e+00
50%      4.000000e+00
75%      5.000000e+00
max      5.000000e+00
Name: rate_clean, dtype: float64
Skewness (should now be small, not 323.71): -1.111
Naive mode-baseline MAE: 1.009
Naive mean-baseline MAE/RMSE: 0.880 / 1.147
Any model you build should meaningfully beat these.


In [12]:
# ------------------------------------------------------------------
# 7. Stratified sampling — no groupby-apply FutureWarning, keeps all cols
# ------------------------------------------------------------------
def stratified_cap(df, col, max_n, random_state=42):
    parts = [g.sample(min(len(g), max_n), random_state=random_state)
              for _, g in df.groupby(col)]
    return (pd.concat(parts)
              .sample(frac=1, random_state=random_state)
              .reset_index(drop=True))

pilot_sample = stratified_cap(merged, 'rate_clean', max_n=10000)
print(f"\nPilot sample (capped at 10k per rate value): {len(pilot_sample):,} rows")


Pilot sample (capped at 10k per rate value): 270,682 rows


In [13]:
# ------------------------------------------------------------------
# 8. Save
# ------------------------------------------------------------------
merged.to_csv('/kaggle/working/comments_merged_clean.csv', index=False, encoding='utf-8-sig')
print("Saved cleaned dataset.")

Saved cleaned dataset.


In [20]:
merged.head(10)

,id,title,body,created_at,rate,recommendation_status,is_buyer,product_id,advantages,disadvantages,...,is_cross_product_dupe,any_quality_flag,advantages_list,disadvantages_list,n_advantages,n_disadvantages,Category1_clean,Category2_clean,sub_category_clean,Brand_clean
0,53672599,پیشنهاد نمیشود,به درد نمیخوره,23 شهریور 1402,1.0,not_recommended,True,252058,NaN,NaN,...,False,False,None,None,0,0,کتاب صوتی,None,book & stationary & art,نوین کتاب گویا
4,53301258,برس رنگ مو,معمولیه اگه واسه خونه رنگ کردن شخصی میخواین او...,12 شهریور 1402,3.0,recommended,True,3255700,NaN,NaN,...,False,False,None,None,0,0,برس ها و تجهیزات آرایشی,تجهیزات رنگ مو,beauty,None
5,53266157,خوبه,قبلا هم استفاده کردم اگه بلد باشین کار کردن با...,11 شهریور 1402,5.0,recommended,True,3305270,NaN,NaN,...,False,False,None,None,0,0,برس ها و تجهیزات آرایشی,برس ها و تجهیزات آرایشی صورت,beauty,None
6,46206235,بدک نیست,خوبه,8 اسفند 1401,2.0,NaN,True,3480048,NaN,NaN,...,False,True,None,None,0,0,برس ها و تجهیزات آرایشی,برس ها و تجهیزات آرایشی صورت,beauty,None
7,40597171,پد شستشوی براش,خیلی به کارتون میاد قیمتشم مناسبه,24 مهر 1401,5.0,recommended,True,3480048,"['جنسش خوبه\r', 'خوش رنگه\r', 'کاربردیه\r', 'ق...",['نداره '],...,False,False,"[جنسش خوبه, خوش رنگه, کاربردیه, قیمت مناسب]",[نداره],4,1,برس ها و تجهیزات آرایشی,برس ها و تجهیزات آرایشی صورت,beauty,None
8,46198656,لنز ناخون,دقیقا مطابق عکس بود,7 اسفند 1401,5.0,recommended,True,7626372,NaN,NaN,...,False,False,None,None,0,0,بهداشت و زیبایی ناخن,آرایش ناخن,beauty,None
9,46901811,خوب,برا حالت دادن با اسپری یا پودر عالیه,20 اسفند 1401,5.0,recommended,True,821812,NaN,NaN,...,False,False,None,None,0,0,برس ها و تجهیزات آرایشی,شانه مو,beauty,None
10,20373536,NaN,اندازش کوچیکه بدرد رنگ مو نمیخوره,14 اردیبهشت 1400,2.0,not_recommended,True,1064806,NaN,NaN,...,False,False,None,None,0,0,برس ها و تجهیزات آرایشی,تجهیزات رنگ مو,beauty,None
11,40710017,عالی,مطابق توضیحات بود,26 مهر 1401,3.0,recommended,True,2727911,NaN,NaN,...,False,False,None,None,0,0,برس ها و تجهیزات آرایشی,تجهیزات رنگ مو,beauty,None
12,53791544,NaN,لب ھارو نرم میکنہ و قیمت مناسب,26 شهریور 1402,5.0,recommended,True,3899330,NaN,NaN,...,False,False,None,None,0,0,آرایش لب,نرم کننده و بالم لب,beauty,ساج


In [15]:
merged.columns

Index(['id', 'title', 'body', 'created_at', 'rate', 'recommendation_status',
       'is_buyer', 'product_id', 'advantages', 'disadvantages', 'likes',
       'dislikes', 'seller_title', 'seller_code', 'true_to_size_rate',
       'Category1', 'Category2', 'sub_category', 'Brand', 'rate_clean',
       'body_token_count', 'is_ultra_short', 'body_text_flag', 'is_promo_spam',
       'is_cross_product_dupe', 'any_quality_flag', 'advantages_list',
       'disadvantages_list', 'n_advantages', 'n_disadvantages',
       'Category1_clean', 'Category2_clean', 'sub_category_clean',
       'Brand_clean'],
      dtype='object')